In [1]:
import torch
import sklearn
import torch.distributions as D
import numpy as np
from numpy.random import default_rng
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pymc3 as pm
import arviz as az

from insilico_stimuli.stimuli import GaborSet

seed = 42
rng = default_rng(seed=seed)
torch.manual_seed(seed=seed)
EPSILON = 1e-12
mpl.style.use("seaborn")
az.style.use("arviz-darkgrid")


/usr/local/lib/python3.8/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 
  warn(f"Failed to load image Python extension: {e}")


In [2]:
import torch

In [ ]:
z_dim = 18
Lambda = torch.ones(z_dim)
sigma = torch.abs(torch.rand(1)) + EPSILON

In [ ]:
gabor_data = torch.load("/src/project/computed/gabors.pt")

In [ ]:
gabor_data

In [ ]:
gabors = gabor_data["gabors"].images()

In [ ]:
gabors.shape

In [ ]:
gabors = torch.tensor(gabors).float()

In [ ]:
(gabors.permute(1, 2, 0) @ D.Exponential(Lambda).sample()).shape

In [ ]:
gabors.shape

In [ ]:
plt.imshow(gabors.sum(dim=0))

In [ ]:
plt.imshow(gabors[9])

In [ ]:
stimulus = gabors[9]

In [ ]:
gabor_params = {
    "canvas_size"         : [50, 50],
    "sizes"               : [15],
    "spatial_frequencies" : [1/20],
    "contrasts"           : [1.0],
    "grey_levels"         : [0.0],
    "eccentricities"      : [0.0],
    "locations"           : [[10, 25], [25, 25], [40, 25]],
    "orientations"        : [np.pi/2],  
    "phases"              : [np.pi/2],
    "relative_sf"         : False
}

gabor_set = GaborSet(**gabor_params)
                     
plt.figure(figsize=(10, 5))
for i, img in enumerate(gabor_set.images()):
    plt.subplot(4, 8, i + 1)
    plt.imshow(img, cmap='gray', vmin=-1, vmax=1)
    plt.axis('off')

In [ ]:
stimuli = torch.tensor(gabor_set.images()).float().sum(dim=0)

In [ ]:
plt.imshow(stimuli)

In [ ]:
model = pm.Model()
with model:
    z = pm.Exponential("neuron", lam=Lambda, shape=z_dim)
    x = pm.Normal("stimulus", mu=(gabors.permute(1, 2, 0) @ z), sigma=sigma, observed=stimuli)
    trace = pm.sample(1_000, return_inferencedata=False)

In [ ]:
trace["neuron"].shape

In [ ]:
with model:
    az.plot_trace(trace)

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=6, dpi=150, sharex=True, sharey=True)
for idx, ax in enumerate(axs.ravel()):
    sns.histplot(trace["neuron"][:, idx].ravel(), color="red", stat="probability", element="step", ax=ax)
    # ax.imshow(gabors[idx], cmap="gray")


In [ ]:
trace["neuron"].shape

In [ ]:
plt.imshow(np.corrcoef(trace["neuron"].T))